# F1 Race Strategy Intelligence — Phase 3: Machine Learning
**Goal:** Predict whether a driver will finish on the podium (top 3) using pre-race and in-race features  
**Model:** Random Forest Classifier with class-weight balancing  
**Sections:**
1. Setup & Feature Engineering
2. Exploratory Feature Analysis
3. Model Training
4. Evaluation (Leave-One-Out CV)
5. Feature Importance
6. Prediction Probabilities
7. What-If Scenario Simulator
8. Honest Limitations & Resume Framing

> **Note on sample size:** We have 60 race entries (3 rounds × 20 drivers).  
> This is intentionally small for now — the pipeline is designed to scale to a full season (460+ entries).  
> All modelling choices account for this — we use LOO-CV and report multiple metrics honestly.

## 1. Setup & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

TEAM_COLORS = {
    'Red Bull Racing': '#3671C6', 'Mercedes': '#6CD3BF', 'Ferrari': '#E8002D',
    'McLaren': '#FF8000', 'Aston Martin': '#358C75', 'Alpine': '#2293D1',
    'Williams': '#37BEDD', 'AlphaTauri': '#5E8FAA',
    'Alfa Romeo': '#C92D4B', 'Haas F1 Team': '#B6BABD',
}
print('Libraries loaded.')

In [ ]:
# ── Load Phase 1 CSVs ─────────────────────────────────────────────────────────
DATA_DIR = 'f1_data'
laps    = pd.read_csv(f'{DATA_DIR}/2023_laps.csv')
pits    = pd.read_csv(f'{DATA_DIR}/2023_pits.csv')
results = pd.read_csv(f'{DATA_DIR}/2023_results.csv')
weather = pd.read_csv(f'{DATA_DIR}/2023_weather.csv')

# ── Feature 1: Number of pit stops per driver per race ───────────────────────
stops = (
    pits.groupby(['EventName', 'Driver'])
    .size()
    .reset_index(name='NumStops')
)

# ── Feature 2: Average weather conditions per race ───────────────────────────
weather['AirTemp']   = pd.to_numeric(weather['AirTemp'],   errors='coerce')
weather['TrackTemp'] = pd.to_numeric(weather['TrackTemp'], errors='coerce')
weather['WindSpeed'] = pd.to_numeric(weather['WindSpeed'], errors='coerce')
weather_avg = (
    weather.groupby('EventName')[['AirTemp', 'TrackTemp', 'WindSpeed']]
    .mean()
    .reset_index()
)

# ── Feature 3: Max tyre life (proxy for longest stint / aggressive undercut) ─
tyre_max = (
    laps.groupby(['EventName', 'Driver'])['TyreLife']
    .max()
    .reset_index(name='MaxTyreLife')
)

# ── Feature 4: Constructor performance (total points = car quality proxy) ────
constructor_pts = (
    results.groupby('TeamName')['Points']
    .sum()
    .reset_index(name='ConstructorPts')
)

# ── Assemble master feature table ────────────────────────────────────────────
df = results[['EventName', 'Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Status']].copy()

# Target variable: podium = finishing P1, P2, or P3
df['Podium'] = (df['Position'] <= 3).astype(int)

df = df.merge(stops,          left_on=['EventName', 'Abbreviation'], right_on=['EventName', 'Driver'], how='left')
df = df.merge(weather_avg,    on='EventName',                                                           how='left')
df = df.merge(tyre_max,       left_on=['EventName', 'Abbreviation'], right_on=['EventName', 'Driver'], how='left')
df = df.merge(constructor_pts, on='TeamName',                                                           how='left')

# Impute missing values
df['NumStops']    = df['NumStops'].fillna(0)
df['MaxTyreLife'] = df['MaxTyreLife'].fillna(df['MaxTyreLife'].median())

# Encode team name as ordinal (by constructor points — best team = highest number)
team_rank = constructor_pts.sort_values('ConstructorPts').reset_index(drop=True)
team_rank['TeamRank'] = team_rank.index + 1
df = df.merge(team_rank[['TeamName', 'TeamRank']], on='TeamName', how='left')

# Binary feature: did they qualify in the top 5?
df['GridTop5']  = (df['GridPosition'] <= 5).astype(int)
df['GridTop10'] = (df['GridPosition'] <= 10).astype(int)

print(f'Feature table: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Podium class balance: {df["Podium"].value_counts().to_dict()} ({df["Podium"].mean()*100:.0f}% podium rate)')
print()
print(df[['Abbreviation', 'EventName', 'GridPosition', 'NumStops', 'MaxTyreLife',
          'ConstructorPts', 'TrackTemp', 'GridTop5', 'Podium']].head(8).to_string(index=False))

## 2. Exploratory Feature Analysis
Before training, check which features visually separate podium from non-podium entries.

In [ ]:
FEATURES = ['GridPosition', 'GridTop5', 'NumStops', 'TrackTemp', 'MaxTyreLife', 'TeamRank', 'ConstructorPts']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Feature distributions — podium vs non-podium', fontsize=14, fontweight='bold')
axes = axes.flatten()

colors = {0: '#95a5a6', 1: '#e74c3c'}
labels = {0: 'No podium', 1: 'Podium'}

for i, feat in enumerate(FEATURES):
    for label in [0, 1]:
        sub = df[df['Podium'] == label][feat].dropna()
        axes[i].hist(sub, bins=12, alpha=0.65, color=colors[label],
                     label=labels[label], density=True, edgecolor='none')
    axes[i].set_title(feat)
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

# Correlation heatmap in last subplot
corr_cols = FEATURES + ['Podium']
corr = df[corr_cols].corr()
sns.heatmap(
    corr[['Podium']].sort_values('Podium', ascending=False),
    annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    ax=axes[7], linewidths=0.5, cbar=False,
    vmin=-1, vmax=1
)
axes[7].set_title('Correlation with podium')
axes[7].set_xlabel('')

plt.tight_layout()
plt.savefig('f1_feature_analysis.png', bbox_inches='tight')
plt.show()

# Print correlation summary
podium_corr = df[corr_cols].corr()['Podium'].drop('Podium').sort_values(key=abs, ascending=False)
print('Feature correlations with Podium (Pearson):')
for feat, val in podium_corr.items():
    direction = 'positive' if val > 0 else 'negative'
    print(f'  {feat:20s}: {val:+.3f}  ({direction})')

## 3. Model Training

**Why Random Forest?**
- Handles mixed feature types (numeric + binary) without scaling
- `class_weight='balanced'` automatically compensates for the 85%/15% class imbalance
- Feature importances are built-in and interpretable
- Robust to the small dataset size better than deep models

**Why `class_weight='balanced'`?**  
Only 15% of entries are podium finishes (9/60). Without balancing, the model would just predict "no podium" every time and get 85% accuracy — which is useless. Balanced weighting penalises missing a podium prediction more heavily.

In [ ]:
X = df[FEATURES].copy()
y = df['Podium'].copy()

# Drop any remaining nulls (should be none after imputation above)
mask  = X.notna().all(axis=1)
X, y  = X[mask], y[mask]

print(f'Training data: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Class balance: {y.value_counts().to_dict()}')
print()

# ── Three models to compare ───────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=0.5, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=3, min_samples_leaf=3,
        class_weight='balanced', random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=50, max_depth=2, learning_rate=0.1, random_state=42
    ),
}

# ── Leave-One-Out Cross Validation ────────────────────────────────────────────
# With only 60 rows, LOO-CV is the most honest evaluation strategy.
# Each fold trains on 59 samples and tests on 1 — uses maximum data.
loo = LeaveOneOut()

results_cv = {}
for name, model in models.items():
    y_pred = cross_val_predict(model, X, y, cv=loo)
    results_cv[name] = {
        'Accuracy':  accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred, zero_division=0),
        'Recall':    recall_score(y, y_pred, zero_division=0),
        'F1':        f1_score(y, y_pred, zero_division=0),
        'y_pred':    y_pred,
    }

cv_df = pd.DataFrame(results_cv).T.drop('y_pred', axis=1).astype(float).round(3)
print('Model comparison — Leave-One-Out CV:')
print(cv_df.to_string())
print()
print('Best model by F1:', cv_df['F1'].idxmax())
print()
print('NOTE: With 60 samples and 15% positive class, these metrics are informative')
print('but should be interpreted cautiously. Expand to full season for robust numbers.')

## 4. Evaluation — Confusion Matrix & Classification Report

In [ ]:
# Use Random Forest as primary model
best_model = models['Random Forest']
y_pred_rf  = results_cv['Random Forest']['y_pred']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Evaluation — Random Forest (LOO-CV)', fontsize=14, fontweight='bold')

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Podium', 'Podium'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion matrix (LOO-CV)')

# ── Per-class metrics bar chart ───────────────────────────────────────────────
report = classification_report(y, y_pred_rf, output_dict=True, zero_division=0)
classes = ['No Podium (0)', 'Podium (1)']
keys    = ['0', '1']
metrics = ['precision', 'recall', 'f1-score']
x = np.arange(len(metrics))
width = 0.3
for i, (cls, key) in enumerate(zip(classes, keys)):
    vals = [report[key][m] for m in metrics]
    axes[1].bar(x + i * width, vals, width, label=cls,
                color=['#95a5a6', '#e74c3c'][i], alpha=0.85, edgecolor='none')
axes[1].set_xticks(x + width / 2)
axes[1].set_xticklabels(['Precision', 'Recall', 'F1-score'])
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Score')
axes[1].set_title('Per-class metrics')
axes[1].legend()
for bar in axes[1].patches:
    h = bar.get_height()
    if h > 0.01:
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.02,
                     f'{h:.2f}', ha='center', fontsize=8)

# ── Predicted probability distribution ───────────────────────────────────────
# Fit RF on full data to get probabilities
best_model.fit(X, y)
df['PodiumProb'] = best_model.predict_proba(X)[:, 1]

for label, color, name in [(0, '#95a5a6', 'No podium'), (1, '#e74c3c', 'Podium')]:
    sub = df[df['Podium'] == label]['PodiumProb']
    axes[2].hist(sub, bins=10, alpha=0.7, color=color, label=name,
                 density=True, edgecolor='none')

axes[2].axvline(0.5, color='black', linestyle='--', linewidth=1, label='Decision threshold (0.5)')
axes[2].set_xlabel('Predicted podium probability')
axes[2].set_ylabel('Density')
axes[2].set_title('Predicted probability distribution')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('f1_model_eval.png', bbox_inches='tight')
plt.show()

print('Classification Report (LOO-CV predictions):')
print(classification_report(y, y_pred_rf, target_names=['No Podium', 'Podium'], zero_division=0))

## 5. Feature Importance

In [ ]:
# ── Two types of importance ───────────────────────────────────────────────────
# 1. Built-in RF impurity-based importance (fast, slightly biased toward high-cardinality features)
# 2. Permutation importance (model-agnostic, more reliable but slower)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')

# ── Gini / impurity importance ────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=True)

bar_colors = ['#e74c3c' if imp > 0.15 else '#3498db' if imp > 0.08 else '#95a5a6'
              for imp in importance_df['Importance']]
axes[0].barh(importance_df['Feature'], importance_df['Importance'],
             color=bar_colors, edgecolor='none', height=0.6)
axes[0].set_xlabel('Mean decrease in impurity')
axes[0].set_title('RF feature importance (Gini)')
for i, (_, row) in enumerate(importance_df.iterrows()):
    axes[0].text(row['Importance'] + 0.002, i, f'{row["Importance"]:.3f}', va='center', fontsize=9)

# ── Permutation importance ────────────────────────────────────────────────────
perm = permutation_importance(best_model, X, y, n_repeats=30, random_state=42, scoring='f1')
perm_df = pd.DataFrame({
    'Feature': FEATURES,
    'Mean':    perm.importances_mean,
    'Std':     perm.importances_std
}).sort_values('Mean', ascending=True)

axes[1].barh(
    perm_df['Feature'], perm_df['Mean'],
    xerr=perm_df['Std'], color='#2ecc71', edgecolor='none',
    height=0.6, error_kw={'ecolor': '#27ae60', 'linewidth': 1.2}
)
axes[1].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Mean decrease in F1 when feature is shuffled')
axes[1].set_title('Permutation importance (F1, 30 repeats)')

plt.tight_layout()
plt.savefig('f1_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top features by permutation importance:')
top = perm_df.sort_values('Mean', ascending=False).head(5)
for _, row in top.iterrows():
    print(f'  {row["Feature"]:20s}: {row["Mean"]:+.4f} ± {row["Std"]:.4f}')
print()
print('Insight: GridPosition + ConstructorPts dominate — qualifying and car quality')
print('are the two strongest predictors of reaching the podium.')

## 6. Prediction Probabilities — Who Did the Model Back?

In [ ]:
# ── Show predicted probabilities alongside actual outcome ─────────────────────
pred_table = df[['Abbreviation', 'TeamName', 'EventName', 'GridPosition',
                  'NumStops', 'Podium', 'PodiumProb']].copy()
pred_table = pred_table.sort_values('PodiumProb', ascending=False)
pred_table['Predicted'] = (pred_table['PodiumProb'] >= 0.5).astype(int)
pred_table['Correct']   = (pred_table['Predicted'] == pred_table['Podium'])

print('Top 15 highest predicted podium probabilities:')
print(pred_table.head(15)[['Abbreviation','EventName','GridPosition',
                             'PodiumProb','Podium','Correct']].to_string(index=False))

# ── Chart: probability per driver per race ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
fig.suptitle('Predicted podium probability by race — top 10 drivers', fontsize=13, fontweight='bold')

for ax, event in zip(axes, sorted(df['EventName'].unique())):
    sub = pred_table[pred_table['EventName'] == event].nlargest(10, 'PodiumProb')
    bar_colors = [
        '#2ecc71' if (row['Podium'] == 1 and row['Predicted'] == 1) else
        '#e74c3c' if (row['Podium'] == 1 and row['Predicted'] == 0) else
        '#f39c12' if (row['Podium'] == 0 and row['Predicted'] == 1) else
        '#bdc3c7'
        for _, row in sub.iterrows()
    ]
    bars = ax.barh(sub['Abbreviation'], sub['PodiumProb'],
                   color=bar_colors, edgecolor='none', height=0.6)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Podium probability')
    ax.set_title(event.replace(' Grand Prix', ' GP'))
    for bar, (_, row) in zip(bars, sub.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{row["PodiumProb"]:.2f}', va='center', fontsize=8)

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Correct podium prediction'),
    mpatches.Patch(color='#e74c3c', label='Missed podium (false negative)'),
    mpatches.Patch(color='#f39c12', label='False alarm (false positive)'),
    mpatches.Patch(color='#bdc3c7', label='Correctly predicted no podium'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.08), fontsize=9)

plt.tight_layout()
plt.savefig('f1_predictions.png', bbox_inches='tight')
plt.show()

## 7. What-If Scenario Simulator
Manually set race parameters and see the model's predicted podium probability.

In [ ]:
def predict_podium(grid_pos, grid_top5, num_stops, track_temp,
                   max_tyre_life, team_rank, constructor_pts):
    """
    Predict podium probability for a hypothetical race entry.

    Parameters
    ----------
    grid_pos        : int   — qualifying position (1–20)
    grid_top5       : int   — 1 if qualified P1-P5, else 0
    num_stops       : int   — number of pit stops made
    track_temp      : float — track surface temperature in °C
    max_tyre_life   : float — laps on longest stint
    team_rank       : int   — constructor ranking (1=worst, 10=best)
    constructor_pts : float — team's total points
    """
    scenario = pd.DataFrame([{
        'GridPosition':   grid_pos,
        'GridTop5':       grid_top5,
        'NumStops':       num_stops,
        'TrackTemp':      track_temp,
        'MaxTyreLife':    max_tyre_life,
        'TeamRank':       team_rank,
        'ConstructorPts': constructor_pts,
    }])
    prob = best_model.predict_proba(scenario)[0, 1]
    pred = 'PODIUM' if prob >= 0.5 else 'No podium'
    return prob, pred


# ── Scenario examples ─────────────────────────────────────────────────────────
scenarios = [
    dict(label='VER-style: P1 grid, top team, 2 stops',
         grid_pos=1, grid_top5=1, num_stops=2, track_temp=31,
         max_tyre_life=28, team_rank=10, constructor_pts=142),
    dict(label='Midfield charge: P8 grid, 1 stop undercut',
         grid_pos=8, grid_top5=0, num_stops=1, track_temp=31,
         max_tyre_life=40, team_rank=5, constructor_pts=50),
    dict(label='Back of grid: P18, backmarker team',
         grid_pos=18, grid_top5=0, num_stops=3, track_temp=31,
         max_tyre_life=15, team_rank=2, constructor_pts=8),
    dict(label='Alonso 2023: P5 grid, strong team',
         grid_pos=5, grid_top5=1, num_stops=2, track_temp=31,
         max_tyre_life=35, team_rank=7, constructor_pts=75),
    dict(label='Mystery: P12 grid but many stops',
         grid_pos=12, grid_top5=0, num_stops=4, track_temp=35,
         max_tyre_life=22, team_rank=6, constructor_pts=60),
]

print('What-If Scenario Simulator')
print('=' * 55)
for s in scenarios:
    prob, pred = predict_podium(
        s['grid_pos'], s['grid_top5'], s['num_stops'],
        s['track_temp'], s['max_tyre_life'], s['team_rank'], s['constructor_pts']
    )
    bar = '#' * int(prob * 25)
    print(f'{s["label"]}')
    print(f'  [{bar:<25s}] {prob:.1%} → {pred}')
    print()

# ── Sensitivity: how does probability change as grid position worsens? ────────
probs_grid = [
    predict_podium(pos, int(pos<=5), 2, 31, 28, 10, 142)[0]
    for pos in range(1, 21)
]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 21), probs_grid, marker='o', color='#e74c3c',
        linewidth=2, markersize=5)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Decision boundary')
ax.fill_between(range(1, 21), probs_grid, 0.5,
                where=[p >= 0.5 for p in probs_grid],
                alpha=0.15, color='#2ecc71', label='Predicted podium zone')
ax.set_xlabel('Starting grid position')
ax.set_ylabel('Predicted podium probability')
ax.set_title('How grid position drives podium probability\n(Red Bull-level car, 2-stop strategy)')
ax.set_xticks(range(1, 21))
ax.legend()
plt.tight_layout()
plt.savefig('f1_sensitivity.png', bbox_inches='tight')
plt.show()

## 8. Honest Limitations & Resume Framing
Being transparent about limitations is itself a data analyst skill.

In [ ]:
print('=' * 60)
print('  MODEL SUMMARY & HONEST ASSESSMENT')
print('=' * 60)
print()
print('WHAT THE MODEL DOES WELL')
print('  - Correctly identifies high-probability podium drivers')
print('  - Feature importances align with domain knowledge')
print('    (GridPosition + ConstructorPts = qualifying + car quality)')
print('  - What-if simulator works as an interpretable tool')
print()
print('KNOWN LIMITATIONS (be ready to discuss in interviews)')
print('  - 60 training samples is small; results may not generalise')
print('  - NumStops and MaxTyreLife are post-race features — in reality')
print('    you would only know these before the race from strategy plans')
print('  - No head-to-head driver quality feature (only car/team quality)')
print('  - Safety car, mechanical failures, weather changes are random')
print('    and cannot be predicted — hence a recall ceiling exists')
print()
print('HOW TO EXPAND THIS PROJECT')
print('  1. Run collect_season() for 2018-2024 — gives ~460 entries/season')
print('  2. Add qualifying lap time as a feature (stronger than grid pos)')
print('  3. Add rolling driver form: avg points in last 3 races')
print('  4. Try XGBoost with hyperparameter tuning via GridSearchCV')
print('  5. Shift target to top-5 finish for more positive samples')
print()
print('RESUME BULLET POINTS')
print('  "Built end-to-end F1 ML pipeline: ingested 3-race telemetry via')
print('   FastF1 API, engineered 7 features across lap, pit, and weather')
print('   data, and trained a Random Forest classifier (LOO-CV) to predict')
print('   podium finishes — GridPosition and ConstructorPts emerged as top')
print('   predictors with permutation importance analysis."')